# Projection cube and ECL

This notebook joins the monthly PD, LGD, EAD and discount-factor terms in the production calculation.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

Period ECL = Marginal PD x LGD x EAD x Discount Factor. Scenario ECL is summed first. Loan ECL is the weighted sum of complete scenario ECL values.

In [2]:
loan = query("select loan_id from worked_trace_summary where stage=2").iloc[0,0]
loan

'RM001506'

In [3]:
detail = query(f'''select loan_id,stage,scenario,scenario_weight,future_month,
conditional_pd,survival_probability,marginal_pd,cumulative_pd,
ead,projected_property_value,projected_ltv,lgd,discount_factor,period_ecl
from worked_trace_monthly where loan_id='{loan}' order by scenario,future_month''')
detail.head(24)

,loan_id,stage,scenario,scenario_weight,future_month,conditional_pd,survival_probability,marginal_pd,cumulative_pd,ead,projected_property_value,projected_ltv,lgd,discount_factor,period_ecl
0,RM001506,2,Base,0.5798,1,0.0314,1.0000,0.0314,0.0314,"230,345.6602","591,195.8385",38.9627,0.0825,0.9965,594.3672
1,RM001506,2,Base,0.5798,2,0.0314,0.9571,0.0300,0.0614,"230,069.6917","592,229.6805",38.8481,0.0820,0.9930,562.6223
2,RM001506,2,Base,0.5798,3,0.0314,0.9160,0.0287,0.0902,"229,792.7458","593,315.9624",38.7302,0.0815,0.9894,532.5115
3,RM001506,2,Base,0.5798,4,0.0314,0.8767,0.0275,0.1177,"229,514.8191","594,453.5264",38.6094,0.0809,0.9860,503.9559
4,RM001506,2,Base,0.5798,5,0.0314,0.8391,0.0263,0.1440,"229,235.9081","595,641.2593",38.4856,0.0804,0.9825,476.8798
5,RM001506,2,Base,0.5798,6,0.0314,0.8031,0.0252,0.1692,"228,956.0092","596,878.0911",38.3589,0.0799,0.9790,451.2110
6,RM001506,2,Base,0.5798,7,0.0314,0.7687,0.0241,0.1933,"228,675.1191","598,162.9925",38.2296,0.0793,0.9756,426.8802
7,RM001506,2,Base,0.5798,8,0.0314,0.7357,0.0231,0.2164,"228,393.2341","599,494.9734",38.0976,0.0788,0.9721,403.8213
8,RM001506,2,Base,0.5798,9,0.0319,0.7041,0.0225,0.2389,"228,110.3508","600,873.0813",37.9632,0.0782,0.9687,388.7914
9,RM001506,2,Base,0.5798,10,0.0357,0.6735,0.0240,0.2630,"227,826.4656","602,296.3993",37.8263,0.0777,0.9653,410.5795


In [4]:
detail['recalculated_ecl'] = detail.marginal_pd * detail.lgd * detail.ead * detail.discount_factor
(detail.period_ecl-detail.recalculated_ecl).abs().max()

np.float64(0.0)

In [5]:
detail.groupby(['scenario','scenario_weight'],as_index=False).period_ecl.sum()

,scenario,scenario_weight,period_ecl
0,Base,0.5798,"12,897.8565"
1,Downside,0.1693,"13,346.8613"
2,Upside,0.2509,"12,818.3696"


SQLite views provide separate Base, Upside and Downside outputs while ecl_projection_cube remains the authoritative calculation table.